# Lidando com Valores Ausentes

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('titanic.csv')

In [ ]:
df.head()

,PassageiroId,PlanetaNatal,CrioSono,Cabine,Destino,Idade,VIP,ServicoDeQuarto,PracaAlimentacao,Shopping,Spa,DeckVirtual,Nome,Transportado
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Terra,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Terra,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [ ]:
df.dtypes

,0
PassageiroId,object
PlanetaNatal,object
CrioSono,object
Cabine,object
Destino,object
Idade,float64
VIP,object
ServicoDeQuarto,float64
PracaAlimentacao,float64
Shopping,float64


In [ ]:
df.shape

(8693, 14)

In [ ]:
df.count()

,0
PassageiroId,8693
PlanetaNatal,8492
CrioSono,8476
Cabine,8494
Destino,8511
Idade,8514
VIP,8490
ServicoDeQuarto,8512
PracaAlimentacao,8510
Shopping,8485


In [ ]:
df.isna().sum()

,0
PassageiroId,0
PlanetaNatal,201
CrioSono,217
Cabine,199
Destino,182
Idade,179
VIP,203
ServicoDeQuarto,181
PracaAlimentacao,183
Shopping,208


In [ ]:
X_train, X_test = train_test_split(df, train_size=0.33, random_state=10)

# 1. Imputar estatísticas diferentes.

- Imputar algumas estatísticas desta feature específica:
     - média (média)
     - mediana
     - percentis
     - moda (mais frequente). Também funciona para recursos categóricos
    
- Imputar algumas estatísticas calculadas dentro de grupos categóricos:
     - Imputar a `Idade` ausente com a `Idade` média/mediana de uma pessoa do mesmo `PlanetaNatal`
     - Imputar `Preço` ausente com o preço médio/mediano de um item da mesma `Categoria`
    
> Deve ser feito com extremo **CUIDADO**, caso contrário é fácil de ajustar demais (overfit).

É melhor que a imputação na porção de **teste** dos dados seja feita com base em estatísticas calculadas na porção de **treino**.

---

## 1.1 Estatística sobre a feature

### Computação manual

In [ ]:
idade_media = df.Idade.mean()
idade_media

np.float64(28.82793046746535)

In [ ]:
#df.Idade.fillna(idade_media, inplace=True)
df.fillna({'Idade': idade_media}, inplace=True)

In [ ]:
# Ou se tivermos X_train, X_test

idade_media = X_train.Idade.mean()
idade_media

np.float64(29.00570613409415)

In [ ]:
X_train.Idade = X_train.Idade.fillna(idade_media)
X_test.Idade = X_train.Idade.fillna(idade_media)

### Usando `sklearn.impute` com `sklearn.pipe.Pipeline`

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer = SimpleImputer(strategy='mean')

df.Idade = imputer.fit_transform(df.Idade.values.reshape(-1, 1))

In [ ]:
df.Idade.values.reshape(-1, 1)

array([[39.],
       [24.],
       [58.],
       ...,
       [26.],
       [32.],
       [44.]])

In [ ]:
# Ou se tivermos X_train, X_test

imputer = SimpleImputer(strategy='mean')
imputer.fit(X_train.Idade.values.reshape(-1, 1))

X_train.Idade = imputer.transform(X_train.Idade.values.reshape(-1, 1))
X_test.Idade = imputer.transform(X_test.Idade.values.reshape(-1, 1))

## 1.2 Estatísticas dentro de grupos

### Computação manual

In [ ]:
idade_media_planeta = df.groupby(['PlanetaNatal'])['Idade'].mean()
idade_media_planeta

,Idade
PlanetaNatal,
Europa,34.419664
Marte,29.297203
Terra,26.068232


In [ ]:
df.loc[df.Idade.isna(),'Idade']

,Idade
50,NaN
64,NaN
137,NaN
181,NaN
184,NaN
...,...
8274,NaN
8301,NaN
8374,NaN
8407,NaN


In [ ]:
df[df.Idade.isna()]

,PassageiroId,PlanetaNatal,CrioSono,Cabine,Destino,Idade,VIP,ServicoDeQuarto,PracaAlimentacao,Shopping,Spa,DeckVirtual,Nome,Transportado
50,0052_01,Terra,False,G/6/S,TRAPPIST-1e,NaN,False,4.0,0.0,2.0,4683.0,0.0,Elaney Hubbarton,False
64,0068_01,Marte,False,E/4/S,TRAPPIST-1e,NaN,False,793.0,0.0,2.0,253.0,0.0,Cinst Binie,False
137,0149_01,Terra,True,G/27/S,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,Billya Hubbarrison,True
181,0202_02,Europa,False,A/2/P,55 Cancri e,NaN,False,0.0,2433.0,NaN,878.0,443.0,Vegas Embleng,True
184,0206_01,Europa,False,C/9/S,55 Cancri e,NaN,False,2.0,1720.0,12.0,1125.0,122.0,Nuson Brugashed,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8274,8835_01,Terra,True,G/1425/S,TRAPPIST-1e,NaN,False,0.0,0.0,0.0,0.0,0.0,Shalle Bartines,False
8301,8862_03,Europa,True,C/329/S,TRAPPIST-1e,NaN,False,0.0,0.0,0.0,0.0,0.0,Alchib Myling,True
8374,8956_04,Terra,False,G/1453/P,TRAPPIST-1e,NaN,False,194.0,1.0,10.0,629.0,0.0,Krisa Bonnondry,False
8407,8988_01,Terra,True,G/1448/S,TRAPPIST-1e,NaN,False,0.0,0.0,0.0,0.0,0.0,Maen Fowlesterez,True


In [ ]:
df[df.Idade.isna()].PlanetaNatal

,PlanetaNatal
50,Terra
64,Marte
137,Terra
181,Europa
184,Europa
...,...
8274,Terra
8301,Europa
8374,Terra
8407,Terra


In [ ]:
df[df.Idade.isna()].PlanetaNatal.map(idade_media_planeta)

,PlanetaNatal
50,26.068232
64,29.297203
137,26.068232
181,34.419664
184,34.419664
...,...
8274,26.068232
8301,34.419664
8374,26.068232
8407,26.068232


In [ ]:
df.loc[df.Idade.isna(),'Idade'] = df[df.Idade.isna()].PlanetaNatal.map(idade_media_planeta)

In [ ]:
df.dtypes

,0
PassageiroId,object
PlanetaNatal,object
CrioSono,object
Cabine,object
Destino,object
Idade,float64
VIP,object
ServicoDeQuarto,float64
PracaAlimentacao,float64
Shopping,float64


### Usando `sklearn.impute` com `sklearn.pipe.Pipeline`

Precisamos escrever uma classe Imputer personalizada

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
class CustomImputer(BaseEstimator, TransformerMixin):
    def __init__(self, groupby_col, impute_col, agg):
        """Imputa valores ausentes usando estatística agregada groupby.

        Parâmetros
        ---

        groupby_col - str,
            coluna para agrupar

        impute_col - str,
            coluna para imputar

        agg - função,
            função de agregção
        """
        self.groupby_col = groupby_col
        self.impute_col = impute_col
        self.agg = agg
        self._mapper = None

    def fit(self, X, y=None):
        self._mapper = X.groupby(self.groupby_col)[self.impute_col].apply(self.agg)
        return self

    def transform(self, X):
        resp = X.copy()
        resp.loc[X[self.impute_col].isna(), self.impute_col] = resp[X[self.impute_col].isna()][self.groupby_col].map(self._mapper)
        return resp

In [ ]:
df.isna().sum()

,0
PassageiroId,0
PlanetaNatal,201
CrioSono,217
Cabine,199
Destino,182
Idade,0
VIP,203
ServicoDeQuarto,181
PracaAlimentacao,183
Shopping,208


In [ ]:
# Estatística sobre a feature

imputer = CustomImputer(groupby_col='PlanetaNatal', impute_col='Idade', agg=np.mean)

novoDF = imputer.fit_transform(df)

In [ ]:
novoDF.head()

,PassageiroId,PlanetaNatal,CrioSono,Cabine,Destino,Idade,VIP,ServicoDeQuarto,PracaAlimentacao,Shopping,Spa,DeckVirtual,Nome,Transportado
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Terra,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Terra,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [ ]:
novoDF.loc[novoDF.Idade.isna(),:]

,PassageiroId,PlanetaNatal,CrioSono,Cabine,Destino,Idade,VIP,ServicoDeQuarto,PracaAlimentacao,Shopping,Spa,DeckVirtual,Nome,Transportado


# 2. Crie uma coluna de indicadores.

Às vezes (frequentemente) o valor ausente de alguma feature é em si um ótimo sinal.
> Lembra a feature `Life Boat` no conjunto de dados do Titanic, o fato de a pessoa ter um valor NA nesta coluna significa que ela provavelmente não sobreviveu (caso contrário, também é verdade, aqueles que não têm um barco salva-vidas não NA provavelmente sobreviveram).

Esta abordagem é especialmente útil se você usar métodos baseados em árvore.

1. Imputar valores de NA com algum valor anômalo e impossível, por ex. valor negativo para `Idade`.

2. Crie uma coluna de indicador adicional, 1 se o valor estiver faltando e 0 caso contrário.

Alternativamente, SE a maioria dos valores estiver faltando, você pode simplesmente criar uma coluna de indicador (e descartar a coluna original).

> Validação cruzada mais fácil, pois poderíamos fazer esse pré-processamento antes da divisão `train-test`.

In [ ]:
df.Idade.isna().astype(int)

,Idade
0,0
1,0
2,0
3,0
4,0
...,...
8688,0
8689,0
8690,0
8691,0


In [ ]:
# Cria coluna de indicador

df['idade_NA'] = df.Idade.isna().astype(int)

# Preenche os valores ausentes com valores impossíveis para esta feature

df.fillna({'Idade':-999}, inplace=True)

# 3. Outros

## 3.1 Não lide com eles.

Algumas das implementações modernas (por exemplo, catboost) irão lidar com valores ausentes para você.
Normalmente, usando um dos métodos acima, por ex. catboost usa duas abordagens:
- “Min” — Os valores ausentes são processados como o valor mínimo (menor que todos os outros valores) para a feature. É garantido que uma divisão que separa os valores faltantes de todos os outros valores seja considerada ao selecionar as árvores.
- “Max” — Os valores ausentes são processados como o valor máximo (maior que todos os outros valores) para a feature. É garantido que uma divisão que separa os valores faltantes de todos os outros valores seja considerada ao selecionar as árvores.

Consuktar https://catboost.ai/docs/concepts/input-data_custom-borders.html

Ambas são versões do `2. Crie uma coluna de indicadores.`

## 3.2 Prever valores ausentes com base em outros recursos (normalmente impraticável)
A ideia é resolver o problema de regressão ou classificação, mas em vez da variável de destino original usar coluna com valores ausentes. O problema é que isso requer validação cruzada incorporada e vai muito além desta disciplina.

## 3.3 Use observações semelhantes (KNN)

> Encontre k observações que tenham representação de características semelhantes (e não NA na coluna obrigatória) e calcule a média delas.

Esta opção é semelhante a `1.2 Estatísticas dentro de grupos`.


In [ ]:
from sklearn.impute import KNNImputer

In [ ]:
# Exemplo da documentação do sklearn

X = np.array([[1, 2, np.nan], [3, 4, 3], [np.nan, 6, 5], [8, 8, np.nan]])
print('Antes:\n', X)

imputer = KNNImputer(n_neighbors=2)
X = imputer.fit_transform(X)
print('\nDepois\n', X)

Antes:
 [[ 1.  2. nan]
 [ 3.  4.  3.]
 [nan  6.  5.]
 [ 8.  8. nan]]

Depois
 [[1.  2.  4. ]
 [3.  4.  3. ]
 [5.5 6.  5. ]
 [8.  8.  4. ]]


In [ ]:
df = pd.read_csv('titanic.csv')

In [ ]:
numer = [col for col in df.columns if df[col].dtypes in ['int64', 'float64']]
numer

['Idade',
 'ServicoDeQuarto',
 'PracaAlimentacao',
 'Shopping',
 'Spa',
 'DeckVirtual']

In [ ]:
df2 = df[numer]

In [ ]:
df2.head()

,Idade,ServicoDeQuarto,PracaAlimentacao,Shopping,Spa,DeckVirtual
0,39.0,0.0,0.0,0.0,0.0,0.0
1,24.0,109.0,9.0,25.0,549.0,44.0
2,58.0,43.0,3576.0,0.0,6715.0,49.0
3,33.0,0.0,1283.0,371.0,3329.0,193.0
4,16.0,303.0,70.0,151.0,565.0,2.0


In [ ]:
imputer = KNNImputer(n_neighbors=5)
df3 = imputer.fit_transform(df2)
df3 = pd.DataFrame(df3, columns=numer)
df3.head()

,Idade,ServicoDeQuarto,PracaAlimentacao,Shopping,Spa,DeckVirtual
0,39.0,0.0,0.0,0.0,0.0,0.0
1,24.0,109.0,9.0,25.0,549.0,44.0
2,58.0,43.0,3576.0,0.0,6715.0,49.0
3,33.0,0.0,1283.0,371.0,3329.0,193.0
4,16.0,303.0,70.0,151.0,565.0,2.0


In [ ]:
df2.isna().sum()

,0
Idade,179
ServicoDeQuarto,181
PracaAlimentacao,183
Shopping,208
Spa,183
DeckVirtual,188


##3.4. Elimine os valores ausentes.

Raramente é uma boa escolha.

In [ ]:
# Elimina linhas com NA

df.Idade.dropna(inplace=True)

In [ ]:
# Elimina colunas com "qualquer" NA

df.dropna(inplace=True, axis=1, how='any')

## Resumo

Eu pessoalmente recomendaria usar `2. Crie uma coluna de indicadores.` quase sempre (ou métodos que funcionam com NA prontos para uso), mas sob certas circunstâncias outros métodos podem funcionar melhor.


---

1. Imputar algum valor estatístico (razoável).
     - Estatística sobre a feature.
     > preencher a Idade NA com a Idade Média
     - Estatística sobre agrupamento de features.
     > Imputar a `Idade` NA com a `Idade` média/mediana das pessoas do mesmo `PlanetaNatal`
2. Imputação com valor anômalo + criação de coluna indicadora (preferencial para métodos baseados em árvore).
3. Outros
     - Deixe-os como estão.
     - Prever valores ausentes com base em outras features (*normalmente* impraticável).
     - Baseado em KNN
     - Apagar (*normalmente* a pior opção).